# Week 10: DPO & Before/After Evals

# Requirements: pip install torch transformers accelerate peft trl bitsandbytes datasets numpy pandas

# ⚠️ REQUIRES: GPU

This notebook creates ~100 **preference pairs** (chosen vs rejected responses), runs a
small **DPO** step via TRL, and produces the **before/after eval table** on a fixed eval
set, including a **catastrophic-forgetting** check on a generic task. The output is the
deploy/no-deploy decision with numbers.


## 0. Setup: repo root on the path + seeded RNG


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import os, json, math, time
import numpy as np
import pandas as pd

from zoro import data

SEED = 42
np.random.seed(SEED)

try:
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig
    from trl import DPOTrainer, DPOConfig
    _TRAIN_OK = True
except Exception as _e:  # pragma: no cover
    torch = None
    _TRAIN_OK = False
    print("training stack not installed:", _e)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print("imports ok, training stack:", _TRAIN_OK)


## 1. System prompts (self-contained copies)

These match the SFT notebook so the eval set is the *same task* before and after.


In [ ]:
EXTRACTION_SYSTEM = (
    "You are ZoroLogistics' document-extraction assistant. Given a bill of lading, "
    "extract the structured fields and return ONLY valid JSON with keys: shipper, "
    "consignee, port_of_loading, port_of_discharge, commodity, quantity, "
    "gross_weight_kg, declared_value_usd, freight_terms, date_of_issue."
)

TRIAGE_SYSTEM = (
    "You are ZoroLogistics' support-router. Classify each ticket into exactly one "
    "category: tracking, damage, refund, documents, customs, or billing. Reply with "
    "only the category word."
)

CATEGORIES = ["tracking", "damage", "refund", "documents", "customs", "billing"]


## 2. The eval set (held out: never used for training)

Three slices graded the same way before and after DPO: 20 held-out bills of lading
(extraction, field-level exact match), 30 held-out tickets (triage accuracy), and 5
generic questions (the forgetting check, the model should already know these).


In [ ]:
eval_bols = data.bol_samples(n=20, seed=777)
eval_tickets = data.support_tickets(n=30, seed=888)

GENERIC_QA = [
    ("What is the capital of France?", "Paris"),
    ("What is 12 * 12?", "144"),
    ("Name a planet in our solar system that starts with M.", "Mars"),
    ("What is the opposite of cold?", "hot"),
    ("What color do you get by mixing red and blue?", "purple"),
]
print("eval set: 20 bols, 30 tickets, 5 generic questions")


## 3. Build ~100 preference pairs

DPO trains on `(prompt, chosen, rejected)`. Here **chosen** is the correct answer and
**rejected** is a deliberately flawed one: an extraction JSON with a corrupted weight, or
a triage answer routed to a *plausible but wrong* category. 50 of each = 100 pairs. In a
real pipeline the rejected side would come from the *actual* model's mistakes, this
hand-written version keeps the notebook deterministic and self-contained.


In [ ]:
def build_preference_pairs(n_bols=50, n_tickets=50, seed=7):
    pairs = []
    bols = data.bol_samples(n=n_bols, seed=11)
    tk = data.support_tickets(n=n_tickets, seed=22)

    for b in bols:
        good = json.dumps(b["fields"], sort_keys=True)
        bad_fields = dict(b["fields"])
        bad_fields["gross_weight_kg"] = int(bad_fields["gross_weight_kg"]) + 1000  # wrong weight
        bad = json.dumps(bad_fields, sort_keys=True)
        pairs.append({
            "prompt": f"{EXTRACTION_SYSTEM}\n\nExtract the fields from this bill of lading:\n{b['text']}",
            "chosen": good,
            "rejected": bad,
        })

    others = {"tracking": "documents", "damage": "refund", "refund": "billing",
              "documents": "customs", "customs": "tracking", "billing": "damage"}
    for _, row in tk.iterrows():
        pairs.append({
            "prompt": f"{TRIAGE_SYSTEM}\n\n{row['text']}",
            "chosen": row["category"],
            "rejected": others[row["category"]],
        })
    return pairs

pref_pairs = build_preference_pairs()
print("preference pairs:", len(pref_pairs))

out_dir = pathlib.Path("dpo_dataset")
out_dir.mkdir(exist_ok=True)
pref_path = out_dir / "dpo_train.jsonl"
with open(pref_path, "w", encoding="utf-8") as f:
    for p in pref_pairs:
        f.write(json.dumps(p) + "\n")
print("saved", pref_path)


## 4. Device + model load

Same base model as the SFT notebook. "Before" = the base model; "After" = the DPO'd
adapter. (In a full pipeline you would DPO the *SFT* adapter; DPO-on-base keeps this
notebook runnable in isolation.)


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def cuda_available():
    try:
        return torch.cuda.is_available()
    except Exception:
        return False

HAS_CUDA = cuda_available()

model = None
tokenizer = None
if not _TRAIN_OK:
    print("Skipping model load, training stack missing.")
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if HAS_CUDA:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
        )
        print("Loaded 4-bit base (QLoRA DPO)")
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, torch_dtype=torch.float16, device_map="mps", trust_remote_code=True
        )
        print("Loaded FP16 base on MPS")
    else:
        print("No CUDA/MPS, model load skipped; evals will return 0.0.")


## 5. Evaluation helpers

Three graders, all deterministic (greedy decoding): triage accuracy, extraction
field-level exact match, and the generic forgetting check.


In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=200):
    if model is None or tokenizer is None:
        return None
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(gen[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def eval_triage(model, tokenizer, df):
    correct, n = 0, 0
    for _, row in df.iterrows():
        out = generate(model, tokenizer, f"{TRIAGE_SYSTEM}\n\n{row['text']}", max_new_tokens=8)
        if out is None:
            continue
        pred = out.split()[0].lower().strip(".,")
        correct += (pred == row["category"])
        n += 1
    return correct / max(n, 1)

EXTRACT_FIELDS = ["shipper", "port_of_loading", "port_of_discharge", "commodity",
                  "gross_weight_kg", "freight_terms"]

def eval_extraction(model, tokenizer, bols):
    total, correct = 0, 0
    for b in bols:
        out = generate(model, tokenizer,
                       f"{EXTRACTION_SYSTEM}\n\nExtract the fields from this bill of lading:\n{b['text']}",
                       max_new_tokens=200)
        if out is None:
            continue
        try:
            start, end = out.find("{"), out.rfind("}")
            parsed = json.loads(out[start:end + 1]) if (start != -1 and end != -1) else {}
        except Exception:
            parsed = {}
        for k in EXTRACT_FIELDS:
            total += 1
            got = str(parsed.get(k, "")).strip().lower()
            want = str(b["fields"][k]).strip().lower()
            correct += (got == want)
    return correct / max(total, 1)

def eval_generic(model, tokenizer, qa):
    correct = 0
    for q, a in qa:
        out = generate(model, tokenizer, f"Answer in one or two words.\nQ: {q}\nA:", max_new_tokens=16)
        if out is None:
            continue
        correct += (a.lower() in out.lower())
    return correct / max(len(qa), 1)


## 6. "Before" eval (base model)

This is the baseline the DPO run must beat, measured on the *same* set we grade later.


In [ ]:
before = {
    "extraction": eval_extraction(model, tokenizer, eval_bols),
    "triage": eval_triage(model, tokenizer, eval_tickets),
    "generic": eval_generic(model, tokenizer, GENERIC_QA),
}
print("BEFORE (base):", {k: round(v, 3) for k, v in before.items()})


## 7. Cost / time estimate (before the DPO cell)

- **Model:** Qwen2.5-1.5B-Instruct, QLoRA DPO (`r=8`), ~0.75 GB weights.
- **Data:** 100 preference pairs, ~40 to 100 tokens each; 1 epoch, batch 1 × grad-accum 4.
- **VRAM:** reference model is cloned → ~2× model memory; still fine on a T4 (16 GB).
- **Time:** a few minutes on a T4. **Cost:** $0 on Colab free tier.
- DPO is heavier than SFT (it keeps a reference model); on a small card, drop `max_length` first.


## 8. Run the DPO step

`beta` controls how strongly we pull toward the chosen response (lower = closer to SFT,
higher = more preference pressure). The run is wrapped so a version/config issue degrades
to a clear message rather than a crash, the eval cells still work either way.


In [ ]:
dpo_ran = False
dpo_model = model
dpo_losses = []

if model is not None and tokenizer is not None:
    dpo_ds = Dataset.from_list(pref_pairs)

    dpo_args = DPOConfig(
        output_dir=str(out_dir / "dpo_checkpoints"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        num_train_epochs=1,
        logging_steps=10,
        fp16=HAS_CUDA,
        report_to=[],
        seed=SEED,
        beta=0.1,
        max_length=512,
        max_prompt_length=256,
    )

    peft_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )

    try:
        trainer = DPOTrainer(
            model=model,
            args=dpo_args,
            beta=0.1,
            train_dataset=dpo_ds,
            tokenizer=tokenizer,
            peft_config=peft_config,
        )
        trainer.train()
        dpo_model = trainer.model
        dpo_ran = True
        for entry in trainer.state.log_history:
            if "loss" in entry:
                dpo_losses.append(entry["loss"])
        print("DPO complete; logged losses:", len(dpo_losses))
    except Exception as _e:
        print("DPO did not run (config/version issue):", _e)
        print("Falling back to base model for the 'after' eval.")
else:
    print("Skipping DPO, no model/tokenizer (GPU required).")


## 9. "After" eval (DPO'd model)

Same three graders, same set. The comparison is only meaningful because nothing about
the eval changed.


In [ ]:
after = {
    "extraction": eval_extraction(dpo_model, tokenizer, eval_bols),
    "triage": eval_triage(dpo_model, tokenizer, eval_tickets),
    "generic": eval_generic(dpo_model, tokenizer, GENERIC_QA),
}
print("AFTER (DPO): ", {k: round(v, 3) for k, v in after.items()})


## 10. The before/after table

The forgetting check is read *against* the gains: a specialist skill is only worth
shipping if the generic drop is small. The decision rule is stated as numbers, not vibes.


In [ ]:
table = pd.DataFrame([
    {"task": "BoL extraction (field F1-ish)", "before": round(before["extraction"], 3),
     "after": round(after["extraction"], 3)},
    {"task": "Ticket triage (accuracy)", "before": round(before["triage"], 3),
     "after": round(after["triage"], 3)},
    {"task": "Generic task (forgetting check)", "before": round(before["generic"], 3),
     "after": round(after["generic"], 3)},
])
table["delta"] = (table["after"] - table["before"]).round(3)
print(table.to_string(index=False))


## 11. Takeaway

DPO sharpens preference without a reward model, but it can also *un-teach* general
skills, the forgetting check exists to catch exactly that. The deploy decision is: did
the specialist deltas earn the generic cost? If not, the honest answer is **don't
deploy**, and stating that with numbers is a passing result.


In [ ]:
# FINAL number: net gain = extraction delta + triage delta - any generic regression.
# 0.0 means nothing ran (or no change); positive = the DPO step earned its cost.
gain = (after["extraction"] - before["extraction"]) + (after["triage"] - before["triage"])
forget = max(0.0, before["generic"] - after["generic"])
net = gain - forget
print(f"BEFORE_AFTER_NET_GAIN={net:.4f}")
print(f"  extraction delta = {after['extraction'] - before['extraction']:+.3f}")
print(f"  triage delta      = {after['triage'] - before['triage']:+.3f}")
print(f"  forgetting cost   = {forget:.3f}")
